In [4]:
import torch
from transformers import MarianMTModel, MarianTokenizer, logging
from datasets import load_dataset
import sacrebleu
import random
logging.set_verbosity_error()
dataset = load_dataset("opus100", "en-fr", split="test")
sample_indices = random.sample(range(len(dataset)), 15)
sampled_data = [dataset[i] for i in sample_indices]
english_sentences = [ex["translation"]["en"] for ex in sampled_data]
french_references = [[ex["translation"]["fr"]] for ex in sampled_data]
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.config.tie_word_embeddings = False  
inputs = tokenizer(
    english_sentences, 
    return_tensors="pt", 
    padding=True, 
    truncation=True, 
    max_length=128
)
with torch.no_grad():
    generated_tokens = model.generate(**inputs)

translated_sentences = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print("\n--- Translations ---\n")
for idx, (src, pred, ref) in enumerate(zip(english_sentences, translated_sentences, french_references), 1):
    print(f"Example {idx}")
    print(f"Source    : {src}")
    print(f"Predicted : {pred}")
    print(f"Reference : {ref[0]}")
    print()
bleu_score = sacrebleu.corpus_bleu(translated_sentences, french_references)
print("BLEU Score:", bleu_score.score)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]


--- Translations ---

Example 1
Source    : he/she/it does not take the air
Predicted : il/elle ne prend pas l'air
Reference : il/elle ne réinvestit pas

Example 2
Source    : CIC offers a number of programs to ease their settlement and integration into Canadian society, including the Immigrant Settlement and Adaptation Program (ISAP), the Language Instruction for Newcomers to Canada (LINC) program and the Host Program, all of which are described as follows.
Predicted : CIC offre un certain nombre de programmes pour faciliter leur établissement et leur intégration dans la société canadienne, y compris le Programme d'établissement et d'adaptation des immigrants (PISA), le Programme d'enseignement linguistique pour les nouveaux arrivants au Canada (LICC) et le Programme d'accueil, qui sont tous décrits comme suit.
Reference : CIC offre plusieurs programmes visant à faciliter leur établissement et leur intégration dans la société canadienne, parmi lesquels le Programme d'établissement et